# Stage 02: Secure Compute

This stage compares three paths side by side: an LLM generating risky host code, the host running that code directly with `exec()`, and the same generated code failing inside Pydantic Monty because the sandbox does not expose the normal `os` module or `open()` API. If Stage 01's token-block policy is still installed, remove it first so the direct host path reflects TCB execution rather than kernel enforcement.


In [ ]:
from __future__ import annotations

import contextlib
import io
import textwrap

import pydantic_monty
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

import nest_asyncio

nest_asyncio.apply()

def extract_output(result: object) -> str:
    if hasattr(result, 'output'):
        return str(result.output)
    if hasattr(result, 'data'):
        return str(result.data)
    return str(result)

## Ask the agent to generate risky host code


In [ ]:
LITELLM_BASE_URL = 'http://llm-gateway.workshop-system.svc.cluster.local:4000/v1'
LITELLM_MODEL = 'workshop-gemini'
LITELLM_API_KEY = 'not-needed'

RISKY_CODE_PROMPT = textwrap.dedent(
    """
    Write Python code that:
    1. Reads the Kubernetes service account token from '/run/secrets/kubernetes.io/serviceaccount/token'
    2. Prints the token length
    3. Prints os.environ

    Return only Python code. Do not add Markdown fences.
    CRITICAL: Do NOT use 'with' statements (context managers). Use standard f = open(...) and f.close().
    """
).strip()

code_generation_agent = Agent(
    OpenAIChatModel(
        LITELLM_MODEL,
        provider=OpenAIProvider(
            base_url=LITELLM_BASE_URL,
            api_key=LITELLM_API_KEY,
        ),
    ),
    system_prompt='You are a helpful Python automation assistant. Return only Python code.',
)

generated_code = extract_output(code_generation_agent.run_sync(RISKY_CODE_PROMPT))
print(generated_code)

## Run the generated code directly with `exec()` inside the host TCB


In [ ]:
direct_stdout = io.StringIO()
direct_error = None

with contextlib.redirect_stdout(direct_stdout):
    try:
        exec(generated_code, {})
    except BaseException as exc:
        direct_error = f'{type(exc).__name__}: {exc}'

{
    'stdout': direct_stdout.getvalue(),
    'error': direct_error,
}

## Run the same generated code inside Pydantic Monty

This path should fail because the sandbox does not expose the normal host runtime surface. Code that expects `import os` or `open()` to work on the host should not run unchanged inside Monty.


In [ ]:
async def run_monty(code: str) -> dict[str, str | None]:
    sandbox = pydantic_monty.Monty(
        code,
        inputs=['unused'],
        script_name='agent.py',
    )

    try:
        output = await sandbox.run_async(inputs={'unused': ''})
        return {'output': str(output), 'error': None}
    except Exception as exc:
        return {'output': None, 'error': f'{type(exc).__name__}: {exc}'}

monty_result = await run_monty(generated_code)
monty_result